# Phase 2b — Deterministic CLAP Embeddings

Revision step 8b from `NEXT_STEPS.md` (PeerJ resubmission).

**Problem.** `laion/larger_clap_music` takes at most 10 s of audio. Its feature extractor is configured with
`truncation='rand_trunc'`, so Phase 2 embedded **one random, unseeded 10-s crop** of each ~30-s MERGE clip.
The audio embeddings were not reproducible, and each track was represented by only a third of its audio.

**Fix.** Split each clip into fixed, non-overlapping 10-s windows (0–10, 10–20, 20–30 s), embed each window,
and mean-pool. A window of exactly 480,000 samples (10 s at 48 kHz) is never cropped. A shorter final window is
repeat-padded, which is also deterministic. Windows shorter than 5 s are dropped. Almost all MERGE clips are
~30 s and get three windows; four tracks are shorter clips (≈10 s or 4.8 s) and get a single window.

Pooling:
* `audio_clap_det` = L2-normalize(mean of L2-normalized window embeddings) — for cosine retrieval and probes;
* `audio_clap_det_raw` = mean of raw window projections — for early fusion, as in Phase 2;
* `audio_clap_windows` = the per-window raw projections (3 × 512), kept for later analysis.

**Output:** `outputs/audio_features_clap_det.parquet`. The original `audio_features.parquet` is left untouched,
so the two variants can be compared (Phase 5d). Checkpoints every 200 tracks; safe to re-run.

In [1]:
import os
import pickle
import time
import warnings
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import ClapModel, ClapProcessor
from tqdm import tqdm

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count())

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR   = os.path.join(ROOT, 'outputs')
OUT_PATH  = os.path.join(OUT_DIR, 'audio_features_clap_det.parquet')
CKPT_PATH = os.path.join(OUT_DIR, 'audio_features_clap_det_checkpoint.pkl')

SR, WIN, MIN_WIN = 48_000, 480_000, 240_000      # 48 kHz, 10-s windows, drop windows < 5 s
DURATION, BATCH_TRACKS, CKPT_EVERY = 30, 4, 200

df = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
audio_df = df[df['has_audio']].reset_index(drop=True)

clap_model = ClapModel.from_pretrained('laion/larger_clap_music').eval()
clap_proc  = ClapProcessor.from_pretrained('laion/larger_clap_music')
fe = clap_proc.feature_extractor
print(f'Tracks: {len(audio_df)} | extractor truncation={fe.truncation}, padding={fe.padding}, '
      f'max {fe.max_length_s}s | fusion={clap_model.config.audio_config.enable_fusion}')

Tracks: 3232 | extractor truncation=rand_trunc, padding=repeatpad, max 10s | fusion=False


## 1. Windowed extraction

In [2]:
def windows(y):
    w = [y[i:i + WIN] for i in range(0, len(y), WIN)]
    w = [x for x in w if len(x) >= MIN_WIN]
    return w or [y]

def embed_windows(wins):
    # Raw (un-normalized) CLAP projections for a list of <=10-s windows, shape (n, 512)
    x = clap_proc(audio=wins, sampling_rate=SR, return_tensors='pt')
    with torch.no_grad():
        out = clap_model.audio_model(input_features=x['input_features'], is_longer=x.get('is_longer'))
        return clap_model.audio_projection(out[1]).numpy().astype(np.float32)

def l2(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# Determinism check on two tracks
y0 = librosa.load(audio_df.audio_path[0], sr=SR, mono=True, duration=DURATION)[0]
e1, e2 = embed_windows(windows(y0)), embed_windows(windows(y0))
print(f'windows for track 0: {[len(w)/SR for w in windows(y0)]} s | identical on repeat: {np.allclose(e1, e2, atol=1e-6)}')

windows for track 0: [10.0, 10.0, 10.0] s | identical on repeat: True


In [3]:
results = {}
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH, 'rb') as fh:
        results = pickle.load(fh)
print(f'Resuming with {len(results)} tracks done')

todo = audio_df[~audio_df.song_id.isin(results)].reset_index(drop=True)
errors, t0 = [], time.time()
for start in tqdm(range(0, len(todo), BATCH_TRACKS), desc='CLAP windows', mininterval=30):
    batch = todo.iloc[start:start + BATCH_TRACKS]
    wins, owner = [], []
    for _, r in batch.iterrows():
        try:
            y = librosa.load(r.audio_path, sr=SR, mono=True, duration=DURATION)[0]
            w = windows(y); wins += w; owner += [r.song_id] * len(w)
        except Exception as e:
            errors.append((r.song_id, str(e)))
    if not wins:
        continue
    E = embed_windows(wins); owner = np.array(owner)
    for sid in dict.fromkeys(owner):
        Ew = E[owner == sid]
        results[sid] = {'audio_clap_windows': Ew,
                        'audio_clap_det_raw': Ew.mean(0),
                        'audio_clap_det': l2(l2(Ew).mean(0)),
                        'n_windows': len(Ew)}
    if len(results) // CKPT_EVERY != (len(results) - len(batch)) // CKPT_EVERY:
        with open(CKPT_PATH, 'wb') as fh:
            pickle.dump(results, fh)

with open(CKPT_PATH, 'wb') as fh:
    pickle.dump(results, fh)
print(f'Done: {len(results)} tracks, {len(errors)} errors, {time.time() - t0:.0f}s this run')
for e in errors[:10]:
    print('  ', e)

Resuming with 3232 tracks done


CLAP windows: 0it [00:00, ?it/s]

CLAP windows: 0it [00:00, ?it/s]

Done: 3232 tracks, 0 errors, 0s this run


## 2. Assemble and save

In [4]:
feat = pd.DataFrame([{'song_id': s,
                      'audio_clap_det': r['audio_clap_det'].tolist(),
                      'audio_clap_det_raw': r['audio_clap_det_raw'].tolist(),
                      'audio_clap_windows': r['audio_clap_windows'].ravel().tolist(),
                      'n_windows': r['n_windows']} for s, r in results.items()])
feat = audio_df[['song_id']].merge(feat, on='song_id', how='left')          # Phase 2 row order
assert feat['audio_clap_det'].notna().all(), 'missing tracks'
X = np.vstack(feat.audio_clap_det.values)
assert X.shape == (len(audio_df), 512) and not np.isnan(X).any()
print('windows per track:', feat.n_windows.value_counts().to_dict())
feat.to_parquet(OUT_PATH, index=False)
print(f'Saved {OUT_PATH} ({os.path.getsize(OUT_PATH)/1e6:.1f} MB)')

windows per track: {3: 3228, 1: 4}


Saved <ROOT>\outputs\audio_features_clap_det.parquet (51.4 MB)


## 3. How unstable was the original random crop?

* Cosine between the old Phase 2 embedding and the new deterministic one, per track.
* Window-to-window cosine within a track: how different the three 10-s windows of one clip look to CLAP.
  This is the spread a single random crop sampled from.
* Rank stability: overlap of each track's top-10 audio neighbours under old vs new embeddings.

In [5]:
old = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features.parquet'))[['song_id', 'audio_clap']]
old = old.set_index('song_id').loc[feat.song_id]
X_old = l2(np.vstack(old.audio_clap.values).astype(np.float64))
X_new = l2(X.astype(np.float64))

cos_old_new = (X_old * X_new).sum(1)
three = (feat.n_windows == 3).values                 # window statistics use the 3-window tracks only
W = l2(np.stack([np.asarray(v, dtype=np.float64).reshape(-1, 512) for v in feat.audio_clap_windows[three]]))
pair = [(0, 1), (0, 2), (1, 2)]
cos_win = np.full((len(feat), 3), np.nan)
cos_win[three] = np.stack([(W[:, i] * W[:, j]).sum(1) for i, j in pair], 1)
best_win = np.stack([(W[:, i] * X_old[three]).sum(1) for i in range(3)], 1).max(1)

print(f'cos(old, new deterministic):   median {np.median(cos_old_new):.3f}  '
      f'[5th pct {np.percentile(cos_old_new, 5):.3f}]')
print(f'cos between windows of a track: median {np.nanmedian(cos_win):.3f}  '
      f'[5th pct {np.nanpercentile(cos_win, 5):.3f}]  (n={three.sum()} tracks with 3 windows)')
print(f'max cos(old, any single window): median {np.median(best_win):.3f}  '
      '(≈1 would mean the old crop matched one fixed window)')

def topk(X, k=10):
    S = X @ X.T; np.fill_diagonal(S, -np.inf)
    return np.argpartition(-S, k, axis=1)[:, :k]
t_old, t_new = topk(X_old), topk(X_new)
overlap = np.array([len(set(a) & set(b)) / 10 for a, b in zip(t_old, t_new)])
print(f'top-10 neighbour overlap old vs new: mean {overlap.mean():.3f}')

pd.DataFrame({'song_id': feat.song_id, 'cos_old_new': cos_old_new,
              'cos_window_mean': np.nanmean(cos_win, 1), 'top10_overlap_old_new': overlap}) \
  .to_csv(os.path.join(OUT_DIR, 'clap_det_stability.csv'), index=False)
print('Saved: clap_det_stability.csv')

cos(old, new deterministic):   median 0.996  [5th pct 0.973]
cos between windows of a track: median 0.991  [5th pct 0.949]  (n=3228 tracks with 3 windows)
max cos(old, any single window): median 0.996  (≈1 would mean the old crop matched one fixed window)


top-10 neighbour overlap old vs new: mean 0.247
Saved: clap_det_stability.csv
